# Lab 8: an LLM judge you can trust

Grade answers with a rubric, then test the judge against labels a person wrote, on defects
you planted, and with each pair shown in both orders. This notebook follows
[Lab 8 in the book](https://handbook.aggarwal.dev/lab-judge.html): read each step there, then run it here. The
code is in `labs/08-judge/judge.py`. On the fake backend a keyword heuristic stands in for
the model, so the numbers measure the plumbing, not a model.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://handbook.aggarwal.dev/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: a task, and labels you trust

Look at one planted defect: the good answer and its copy with a wrong number, and the label
a person gave each. The labels are the truth every number below is measured against.
[Step 1](https://handbook.aggarwal.dev/lab-judge.html#step-1) lists the four defect types.

In [ ]:
sys.path.insert(0, str(root / "labs" / "08-judge"))
import judge
from common import llm

items = judge.load_items()
print(len(items), "labelled answers")
for item in items[:2]:
    print(item["id"], item["human_label"], item.get("defect"), "|", item["question"], "|", item["answer"])

## Step 2: the judge prompt

The rubric has four tests, the reason comes before the verdict, and the reply is one JSON
object. Print the system prompt and the message the judge receives for one answer.

In [ ]:
print(judge.JUDGE_SYSTEM)
print()
print(judge.judge_prompt(items[0]["question"], items[0]["answer"])[-300:])

## Step 3: parse strictly, count what does not parse

`parse_verdict` accepts exactly one JSON object with a `reason` and a `verdict`, and
returns `None` for anything else. Try the replies the page lists; only the first is accepted.

In [ ]:
replies = {
    "good": '{"reason": "all four tests hold", "verdict": "PASS"}',
    "prose first": 'Here you go: {"reason": "ok", "verdict": "PASS"}',
    "code fence": '```json\n{"reason": "ok", "verdict": "PASS"}\n```',
    "lower case": '{"reason": "ok", "verdict": "pass"}',
    "no reason": '{"verdict": "PASS"}',
    "extra key": '{"reason": "ok", "verdict": "PASS", "score": 9}',
}
for name, text in replies.items():
    print(f"{name:12}", judge.parse_verdict(text))

## Step 4: agreement and Cohen's kappa

The page's worked example: seven of ten verdicts agree, yet kappa is only 0.348. This is the
page's own command.

In [ ]:
!python labs/08-judge/judge.py kappa-example

Call `cohen_kappa` yourself. Perfect agreement gives 1, and a rater who gives one label to
everything makes kappa undefined (`None`).

In [ ]:
person = list("PPPPPPFFFF")
print(judge.cohen_kappa(person, list("PPPPPFFFPP")))
print(judge.cohen_kappa(person, person))
print(judge.cohen_kappa(list("PPPP"), list("PPPP")))

## Step 5: which way does the judge err?

The full run below prints the confusion matrix, recall per defect type and the position-bias
count. **Cost:** it makes 80 calls. On `claude-haiku-5-5` the page puts that at about
$0.0035, and at most $0.0848 if every reply used its whole `max_tokens`. The fake backend and
Ollama cost nothing. Look for the one unparseable reply, and which defect types are missed.

In [ ]:
!LLM_BACKEND=fake python labs/08-judge/judge.py run

## Step 6: position bias in a pairwise judge

Each (good, bad) pair was judged twice, once in each order. `position_flips` maps both
replies back to the answer chosen and counts the pairs where the order decided. Try it on
made-up replies: a "1" then a "2" means the same answer won both times.

In [ ]:
print(judge.position_flips(["1", "1", "2", None], ["2", "1", "1", "2"]))

## Run the tests

The tests need no key and no network.

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 08-judge

## Experiments

Try the experiments in the [Experiments section](https://handbook.aggarwal.dev/lab-judge.html#experiments). A quick one:
edit the rubric in the cell below to a vague one and re-run the page's `run` command on a
real model, then compare kappa. On the fake backend the heuristic ignores the rubric, so
nothing changes there.

In [ ]:
judge.JUDGE_SYSTEM = judge.JUDGE_SYSTEM.split("The answer passes only")[0] + (
    'Is this a good answer? Reply with one JSON object and nothing else:\n'
    '{"reason": "one short sentence", "verdict": "PASS" or "FAIL"}'
)
print(judge.JUDGE_SYSTEM)

Then tick the lab's checklist in the book: your progress is kept in your browser, not in
this notebook.